# Strike — research notes

Thomas Ou · November 2026

Strike labels short MMA video windows as **strike attempt** or **no attempt**. A committed strike is positive even when it misses or is blocked. The website contains the [annotation workspace](https://strike.thomasou.com/workspace) and [recorded demo](https://strike.thomasou.com/demo); this notebook keeps the experiment record alongside the code.

**Current evidence:** a 38-window pilot, historical TSN training logs, and recorded predictions from a separate frozen-ResNet development baseline. There is **no independent test result** or established accuracy on unseen fights.

Run all cells with Python 3.11+ from this checkout. They use the standard library, read committed files, and verify their provenance. They do not train a model, download footage, or require PyTorch.

## Research question

Can a five-frame appearance baseline distinguish committed strike attempts on held-out fights, and does foreground masking improve that comparison?

**Working hypothesis:** masking may reduce reliance on broadcast backgrounds, but it may also remove useful context or introduce segmentation errors. The direction and size of its effect remain unmeasured. The comparison changes the input representation while holding the reviewed windows, labels, fight assignments, encoder, sampling policy, and optimization settings fixed.

The outcome is a binary **window label**, not a count of strikes or a landed-strike judgment. A fight is the grouping unit for data separation; neighboring windows are dependent observations. Five sampled frames do not imply a fixed duration, so interval length and sampling policy must be recorded alongside the labels.

In [1]:
from pathlib import Path
import hashlib
import json
import math
import runpy
import statistics

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "model/export_notebook_history.py").is_file()
    and (path / "showcase/src/lib/recorded-demo.json").is_file()
)
history_path = ROOT / "showcase/src/lib/legacy-training.json"
demo_path = ROOT / "showcase/src/lib/recorded-demo.json"
history = json.loads(history_path.read_text())
demo = json.loads(demo_path.read_text())

for path in (history_path, demo_path):
    print(path.relative_to(ROOT))
    print(f"  SHA-256: {hashlib.sha256(path.read_bytes()).hexdigest()}")

showcase/src/lib/legacy-training.json
  SHA-256: 992bf6752ef59ac2f7db1354fe0a6c70da69015c49594ef049234761ac99b748
showcase/src/lib/recorded-demo.json
  SHA-256: 13b53ecad67124f6034907e947bad55e2da57dcf02f7b5c1e3752d1cff2d52f0


## Data and label definition

The original collection has **38 unique five-frame windows** (190 frames): 19 labeled strike and 19 neutral. Those labels predate the current attempt definition and need review.

| Original partition | Windows | Interpretation |
| :--- | ---: | :--- |
| Training | 30 | Development data |
| Validation | 6 | Inspected during model selection |
| Test | 2 | Both also occur in validation |
| Unused | 2 | Outside the original splits |

These memberships overlap; the rows must not be added as disjoint partitions. The legacy import now retains their memberships while assigning all examples to development. Fight identifiers and timestamps were not recorded. A sampled frame identifies Gaethje–Holloway; it does not establish the identity of every source clip.

**Attempt:** a committed strike, regardless of contact. **No attempt:** movement, guard adjustments, feints, or clinching alone. Keep occluded or ambiguous examples **uncertain** until reviewed; they are excluded from binary training labels. Contact outcome is a separate annotation.

Five frames need not contain a whole action. At an assumed 30 fps they occupy about 167 ms of playback, with 133 ms between the first and last sample. Actual source timing must come from the video; neither this assumption nor preview speed supplies missing timestamps. Window predictions also cannot be counted as unique strikes without event boundaries.

The [manifest and labeling reference](../../PROJECT_OVERVIEW.md) specifies source fight, interval, split, frame paths, and checksums. Assign entire fights to partitions before extracting windows, and keep test fights uninspected during model selection.

## Evidence available

| Evidence | What can be inspected | What it does not establish |
| :--- | :--- | :--- |
| Historical TSN run | Saved 20-epoch training and validation traces | Independent test performance; a comparison with the current baseline |
| Frozen-ResNet demo | 120 saved predictions, 600 input-frame hashes, and recorded checkpoint provenance | Label accuracy, calibrated probabilities, or counts of unique strikes |
| Raw versus masked inputs | Training, evaluation, and paired-comparison code | A measured benefit from masking; no matched result is recorded |

The executed cells below reproduce saved summaries and check their inputs. Training and final evaluation require a separately reviewed dataset; the command examples at the end are an experiment procedure, not results from an executed benchmark.

## Historical TSN run

These are saved **MMAction2 TSN** metrics ([2](https://arxiv.org/abs/1608.00859)), not results for the newer frozen-backbone baseline. The exporter reads the original notebook output and records its SHA-256 and source cell indices. The following cell checks both the fingerprint and the complete exported history before displaying any metrics.

In [2]:
original_notebook = ROOT / history["source"]
assert hashlib.sha256(original_notebook.read_bytes()).hexdigest() == history["source_sha256"]
export_history = runpy.run_path(str(ROOT / "model/export_notebook_history.py"))["export_history"]
assert export_history(original_notebook) == history

print(f"Source: {history['source']}")
print(f"Output cells (zero-based): {history['cell_indices']}")
print(history["note"])
print()
print(f"{'Epoch':>5} {'Train loss':>12} {'Train accuracy':>16} {'Validation accuracy':>21}")
for epoch in history["epochs"]:
    print(f"{epoch['epoch']:5d} {epoch['train_loss']:12.4f} "
          f"{epoch['train_accuracy']:16.2%} {epoch['val_accuracy']:21.2%}")

Source: model/notebooks/TSN_Pipeline.ipynb
Output cells (zero-based): [6]
Historical validation logs, not a held-out benchmark. Both legacy test clips also appear in validation. No independent test result is recorded.

Epoch   Train loss   Train accuracy   Validation accuracy
    1       0.7090           46.67%               100.00%
    2       0.7002           60.00%               100.00%
    3       0.6881           73.33%                50.00%
    4       0.6810           63.33%                50.00%
    5       0.6676           83.33%                50.00%
    6       0.6525           90.00%                33.33%
    7       0.6375           86.67%                33.33%
    8       0.6192           96.67%                33.33%
    9       0.6028           93.33%                33.33%
   10       0.5843           96.67%                33.33%
   11       0.5645           96.67%                33.33%
   12       0.5441           96.67%                33.33%
   13       0.5227         

In [3]:
epochs = history["epochs"]
final = epochs[-1]
best_validation = max(epoch["val_accuracy"] for epoch in epochs)
best_epochs = [epoch["epoch"] for epoch in epochs if epoch["val_accuracy"] == best_validation]
print(f"Final epoch: {final['epoch']}")
print(f"Final train / validation accuracy: {final['train_accuracy']:.1%} / {final['val_accuracy']:.1%}")
print(f"Highest recorded validation accuracy: {best_validation:.1%}, epochs {best_epochs}")
print("Validation size: six clips. Independent test accuracy: not recorded.")

Final epoch: 20
Final train / validation accuracy: 100.0% / 50.0%
Highest recorded validation accuracy: 100.0%, epochs [1, 2]
Validation size: six clips. Independent test accuracy: not recorded.


The final epoch reaches **100% training accuracy and 50% validation accuracy**. Earlier 100% validation epochs describe six clips, not unseen-fight performance. The overlapping historical test set cannot resolve that limitation.

The original notebook uses ImageNet ResNet-50 weights without configuring the backbone as frozen. Its configuration does not fix the random seed, and its inherited test pipeline differs from validation. A downloaded Kinetics checkpoint is **not loaded** (`load_from=None`). The saved best checkpoint is from epoch 1, while the inference cell loads epoch 20.

The original [TSN notebook](TSN_Pipeline.ipynb) and [3D-CNN notebook](3DCNN_Pipeline.ipynb) remain historical artifacts. The latter is not a controlled comparison against the current baseline.

## Current appearance baseline

The implementation in [baseline.py](../baseline.py) freezes an ImageNet-1K V1 ResNet-50 ([1](https://arxiv.org/abs/1512.03385)), averages its frame features, and trains a binary linear head. Batch normalization in the encoder stays in evaluation mode.

| Setting | Value |
| :--- | :--- |
| Input | Five RGB frames |
| Preprocessing | Resize short edge to 256, center crop to 224, ImageNet normalization |
| Trainable parameters | Two-class linear head |
| Aggregation | Mean of the five frame feature vectors |
| Temporal order | Not modeled; permuting frames leaves the mean unchanged |

This can learn visual poses and context, but does not explicitly represent motion direction or timing. The new training pipeline records seed, preprocessing, dataset/source fingerprints, and checkpoint provenance. Research runs select on validation loss and evaluate the frozen choice separately on test data; development smoke runs are not benchmarks.

**Segmentation is an experiment.** SAM2 ([3](https://arxiv.org/abs/2408.00714)) overlays demonstrate a possible input variant, not an established improvement in detection. A raw/masked comparison needs the same reviewed examples, fight splits, seed, preprocessing, and training settings, with checkpoint selection confined to validation.

## Recorded demo predictions

The demo uses `recorded-demo.json`, not illustrative scores. It contains offline outputs from a **separate frozen-ResNet development checkpoint trained on the 38 pilot windows**. Its model scores are uncalibrated outputs; the JSON field named `confidence` does not make them calibrated confidence estimates.

The three displayed sequences have no reviewed ground-truth labels, source timestamps, or benchmark metrics. The visible scoreboard reads Herbert / Topuria, but the acquisition history and original video were not recorded. Cuts and replays may occur. Preview playback speed is not source timing.

The table below summarizes saved outputs. “Positive windows” means scores at or above the recorded threshold; it is **neither accuracy nor a count of strikes**.

In [4]:
model = demo["model"]
provenance = demo["provenance"]
threshold = model["threshold"]
assert demo["kind"] == "recorded_checkpoint_predictions"
assert provenance["groundTruthLabels"] is None
assert provenance["benchmarkMetrics"] is None

print(f"Architecture: {model['architecture']}")
print(f"Run purpose: {model['purpose']}; input variant: {model['variant']}")
print(f"Training windows: {model['trainingClipCount']}; selected epoch: {model['selectedEpoch']}")
print(f"Recorded threshold: {threshold:.2f}; frame-order sensitive: {model['frameOrderSensitive']}")
print()
print(f"{'Sequence':<12} {'Frames':>6} {'Windows':>8} {'Mean score':>11} {'Min':>8} {'Max':>8} {'Positive windows':>17}")
for name, clip in demo["clips"].items():
    scores = [prediction["confidence"] for prediction in clip["predictions"]]
    assert all(math.isfinite(score) and 0 <= score <= 1 for score in scores)
    positive = sum(score >= threshold for score in scores)
    print(f"{name:<12} {clip['totalFrames']:6d} {len(scores):8d} "
          f"{statistics.mean(scores):11.4f} {min(scores):8.4f} {max(scores):8.4f} {positive:17d}")
print(f"Total saved windows: {sum(len(clip['predictions']) for clip in demo['clips'].values())}")

Architecture: resnet50_frozen_frame_average_linear
Run purpose: development_smoke; input variant: raw
Training windows: 38; selected epoch: 20
Recorded threshold: 0.50; frame-order sensitive: False

Sequence     Frames  Windows  Mean score      Min      Max  Positive windows
knockdown       200       40      0.4142   0.0460   0.8780                13
exchange        200       40      0.4691   0.0056   0.9541                18
pressure        200       40      0.6367   0.1192   0.9540                29
Total saved windows: 120


### Check the displayed inputs

The next cell verifies every displayed frame against the hashes in the saved export and checks the five-frame window indices and threshold-derived labels. It does **not** rerun neural inference or validate label accuracy.

The export also records a previous six-window reproduction check against the original checkpoint. That is a provenance check of the saved predictions, not evidence that the model correctly identifies attempts.

In [5]:
verified_frames = 0
verified_windows = 0
for name, clip in demo["clips"].items():
    assert len(clip["frameSha256"]) == clip["totalFrames"]
    assert len(clip["predictions"]) * clip["windowSize"] == clip["totalFrames"]
    frame_dir = ROOT / "showcase/public" / clip["frameDir"].lstrip("/")
    for offset, expected_hash in enumerate(clip["frameSha256"]):
        file_index = offset + clip["frameFileIndexBase"]
        frame_path = frame_dir / (clip["framePattern"] % file_index)
        assert hashlib.sha256(frame_path.read_bytes()).hexdigest() == expected_hash, frame_path.name
        verified_frames += 1
    for index, prediction in enumerate(clip["predictions"]):
        start = index * clip["windowSize"]
        assert prediction["window"] == index
        assert prediction["startFrame"] == start
        assert prediction["endFrame"] == start + clip["windowSize"] - 1
        expected_label = "strike" if prediction["confidence"] >= threshold else "neutral"
        assert prediction["label"] == expected_label
        verified_windows += 1

print(f"Verified locally: {verified_frames} frame hashes and {verified_windows} window records.")
print(f"Recorded checkpoint SHA-256: {model['checkpointSha256']}")
print(f"Recorded training-manifest SHA-256: {model['trainingManifestSha256']}")
print(f"Previous export reproduction check: {provenance['reproducedWindowCount']} windows;")
print(f"maximum absolute score difference: {provenance['maxReproductionAbsoluteDifference']:.2e}.")
for filename, expected_hash in model["sourceSha256"].items():
    matches = hashlib.sha256((ROOT / "model" / filename).read_bytes()).hexdigest() == expected_hash
    print(f"Current source matches recorded source, {filename}: {matches}")

Verified locally: 600 frame hashes and 120 window records.
Recorded checkpoint SHA-256: 196f5a7fe8ea7c9916068aa20afc6e4fda67862dcf5991e53af2f1613957e4d2
Recorded training-manifest SHA-256: 289ff922ffd2dce394c1f85561675b1abaa04ea1cecbb50ac9dc957ef0d53091
Previous export reproduction check: 6 windows;
maximum absolute score difference: 5.96e-07.
Current source matches recorded source, baseline.py: True
Current source matches recorded source, data.py: True


## Planned comparison and reporting

This is a proposed design for the next experiment, not a preregistration or a completed result. Finalize it on development and validation data before inspecting test predictions.

| Decision | Planned treatment |
| :--- | :--- |
| Comparison | Raw frames versus their paired masked frames, trained separately |
| Fixed inputs | Same reviewed manifest, fight assignments, five sampled frames, and labels |
| Fixed training | Same frozen encoder, preprocessing, seed, optimizer, and training budget |
| Checkpoint selection | Lowest validation loss within each run; selected epochs may differ |
| Primary comparison | Masked minus raw balanced accuracy at the recorded threshold of 0.5 |
| Supporting results | Average precision, precision/recall/F1, log loss, Brier score, and confusion matrices |
| Scope of the estimate | The reserved fights and the documented window-sampling policy |

Report class counts, numbers of windows and fights, and each fight's metrics alongside the pooled result. Balanced accuracy requires both classes; leave it undefined for a single-class fight rather than substituting zero. Pooled metrics weight windows equally, so fights with more windows have more influence. Class prevalence also affects precision and average precision; a balanced collection of selected windows does not represent the prevalence in an entire fight.

The comparison tool checks matched runs and reports which windows change correctness. It produces descriptive differences, not confidence intervals or significance tests. Any later uncertainty analysis must account for dependence within fights; treating neighboring windows as independent would overstate the evidence. Repeat any selected seed comparisons under the same frozen protocol, and report all runs rather than choosing the best test result.

## What remains unmeasured

- **New fights:** independently reviewed footage with fight-level separation and an untouched test partition. More neighboring windows from the same sequence do not establish generalization.
- **Hard negatives and missed attempts:** feints, clinch breaks, fast movement, blocked strikes, and misses need separate evaluation slices.
- **Occlusion and boundaries:** incomplete visibility and attempts crossing windows need reviewed event timestamps before reporting event-level recall or counts.
- **Masking:** a paired experiment is required before attributing any improvement to segmentation.

These are open evaluation questions, not measured failure rates. No demo score, training accuracy, or earlier validation peak answers them.

## Reproduce and extend

All commands below run from the repository root. See the [README](../../README.md) for Python dependencies and complete setup. The executed cells above require no model dependencies.

Regenerate the historical JSON from the saved TSN notebook output:

```bash
python model/export_notebook_history.py
```

For new footage, label windows in the workspace and export its annotation backup. Keep source video outside Git. Prepare the frames and score them using an existing completed raw model run:

```bash
python -m model.prepare --annotations annotations.json --video /path/to/footage.mp4 --output data-local/reviewed
python -m model.predict --prepared data-local/reviewed --run runs/your-raw-run --output runs/workspace-review
```

Import `runs/workspace-review/predictions.json` into the workspace. The browser reviews footage and scores; inference runs locally in Python. Predictions do not replace manual labels. Use new output directories to preserve prior results.

For a research experiment, curate one manifest with reviewed labels, source provenance, paired raw and masked frames, and whole fights assigned to train, validation, and test before training. Use the same environment for both runs and new output paths throughout.

Audit both input variants, then train the matched models:

```bash
python -m model.data audit --manifest data-local/reviewed.json --data-root /path/to/frames --variant raw --output runs/raw-audit.json
python -m model.data audit --manifest data-local/reviewed.json --data-root /path/to/frames --variant masked --output runs/masked-audit.json
python -m model.train --manifest data-local/reviewed.json --data-root /path/to/frames --output runs/raw --variant raw --seed 42
python -m model.train --manifest data-local/reviewed.json --data-root /path/to/frames --output runs/masked --variant masked --seed 42
python -m model.compare --raw runs/raw --masked runs/masked --split validation --output runs/validation-comparison.json
```

Select the design on validation. After freezing the design, checkpoints, and reporting plan, evaluate the same reserved fights for both models:

```bash
python -m model.evaluate --run runs/raw --manifest data-local/reviewed.json --data-root /path/to/frames
python -m model.evaluate --run runs/masked --manifest data-local/reviewed.json --data-root /path/to/frames
python -m model.compare --raw runs/raw --masked runs/masked --split test --output runs/test-comparison.json
```

Each run records its configuration, manifest, audit, training history, selected checkpoint, and per-window predictions. Test outputs live under each run's `evaluation/` directory. The comparison JSON records metric differences and paired correctness outcomes; retain it with both run directories.

The current pilot cannot supply that independent test set. These commands describe the supported workflow; this notebook has not run new training or test evaluation.

## References

1. Kaiming He et al. (2015). [Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385). Backbone used by the current appearance baseline.
2. Limin Wang et al. (2016). [Temporal Segment Networks: Towards Good Practices for Deep Action Recognition](https://arxiv.org/abs/1608.00859). Method underlying the historical TSN experiment.
3. Nikhila Ravi et al. (2024). [SAM 2: Segment Anything in Images and Videos](https://arxiv.org/abs/2408.00714). Segmentation method referenced by the overlay scripts; a raw/masked performance comparison is not yet reported here.